# PySpark DataFrame API và Parquet

Notebook này hướng dẫn thực hiện bài lab trên bộ dữ liệu 100.000 đơn hàng. Chạy lần lượt các cell từ trên xuống dưới. Spark chạy trong container ở chế độ `local[*]`; CSV được đọc từ thư mục `data/` và kết quả Parquet được ghi vào `output/` trong repository.

**Mục tiêu:** schema tường minh, DataFrame transformations, aggregation, Window Function, execution plan, Parquet, partitioning và partition pruning. Hãy hoàn thành các câu hỏi/nhận xét ở những cell TODO.

## 1. Khởi tạo SparkSession

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PySparkDataFrameParquetLab")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

WORKSPACE = Path.cwd()
if not (WORKSPACE / "data/sales_100k.csv").is_file():
    WORKSPACE = WORKSPACE.parent

DATA_PATH = WORKSPACE / "data/sales_100k.csv"
OUTPUT_DIR = WORKSPACE / "output"
print(f"CSV: {DATA_PATH}")
print(f"Output: {OUTPUT_DIR}")
assert DATA_PATH.is_file(), f"CSV not found: {DATA_PATH}"

CSV: /home/jovyan/work/data/sales_100k.csv
Output: /home/jovyan/work/output


## 2. Khai báo schema tường minh

Không bật `inferSchema=True`: schema được định nghĩa trước khi đọc dữ liệu.

In [2]:
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
)

schema = StructType(
    [
        StructField("order_id", LongType(), False),
        StructField("order_date", StringType(), False),
        StructField("customer_id", StringType(), False),
        StructField("customer_segment", StringType(), False),
        StructField("country", StringType(), False),
        StructField("city", StringType(), False),
        StructField("category", StringType(), False),
        StructField("product_id", StringType(), False),
        StructField("quantity", IntegerType(), False),
        StructField("unit_price", DoubleType(), False),
        StructField("discount_pct", IntegerType(), False),
        StructField("payment_method", StringType(), False),
        StructField("status", StringType(), False),
    ]
)
schema

StructType([StructField('order_id', LongType(), False), StructField('order_date', StringType(), False), StructField('customer_id', StringType(), False), StructField('customer_segment', StringType(), False), StructField('country', StringType(), False), StructField('city', StringType(), False), StructField('category', StringType(), False), StructField('product_id', StringType(), False), StructField('quantity', IntegerType(), False), StructField('unit_price', DoubleType(), False), StructField('discount_pct', IntegerType(), False), StructField('payment_method', StringType(), False), StructField('status', StringType(), False)])

## 3. Đọc CSV và kiểm tra dữ liệu

Số dòng dữ liệu mong đợi: **100.000** (không tính header).

In [3]:
df_raw = (
    spark.read
    .option("header", True)
    .option("mode", "FAILFAST")
    .schema(schema)
    .csv(str(DATA_PATH))
)

row_count = df_raw.count()
print(f"Rows: {row_count:,}")
assert row_count == 100_000, f"Expected 100,000 rows, found {row_count:,}"
df_raw.printSchema()
df_raw.show(10, truncate=False)

Rows: 100,000
root
 |-- order_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- status: string (nullable = true)

+--------+----------+-----------+----------------+-------+----------------+-----------+----------+--------+----------+------------+--------------+---------+
|order_id|order_date|customer_id|customer_segment|country|city            |category   |product_id|quantity|unit_price|discount_pct|payment_method|status   |
+--------+----------+-----------+----------------+-------+----------------+-----------+----------+--------+----------+--------

## 4. Chuẩn hóa và tạo cột tính toán

`gross_amount = quantity × unit_price`; `net_amount = gross_amount × (1 − discount_pct / 100)`.

In [4]:
from pyspark.sql import functions as F

df = (
    df_raw
    .withColumn("order_date", F.to_date("order_date", "yyyy-MM-dd"))
    .withColumn(
        "gross_amount",
        F.round(F.col("quantity") * F.col("unit_price"), 2),
    )
    .withColumn(
        "net_amount",
        F.round(
            F.col("gross_amount")
            * (1 - F.col("discount_pct") / F.lit(100.0)),
            2,
        ),
    )
    .withColumn("order_year", F.year("order_date"))
    .withColumn("order_month", F.month("order_date"))
)

df.select(
    "order_id",
    "quantity",
    "unit_price",
    "discount_pct",
    "gross_amount",
    "net_amount",
).show(20, truncate=False)

+--------+--------+----------+------------+------------+----------+
|order_id|quantity|unit_price|discount_pct|gross_amount|net_amount|
+--------+--------+----------+------------+------------+----------+
|1       |1       |134.95    |0           |134.95      |134.95    |
|2       |4       |850.97    |0           |3403.88     |3403.88   |
|3       |1       |1490.04   |0           |1490.04     |1490.04   |
|4       |5       |598.6     |10          |2993.0      |2693.7    |
|5       |5       |1361.71   |0           |6808.55     |6808.55   |
|6       |1       |304.55    |10          |304.55      |274.1     |
|7       |5       |260.69    |0           |1303.45     |1303.45   |
|8       |4       |190.92    |5           |763.68      |725.5     |
|9       |3       |287.34    |0           |862.02      |862.02    |
|10      |2       |64.48     |0           |128.96      |128.96    |
|11      |4       |15.37     |10          |61.48       |55.33     |
|12      |5       |193.37    |5           |966.8

## 5. Lọc và chọn cột

Tìm đơn hàng tại Việt Nam, trạng thái `Completed`, doanh thu ròng từ 500 trở lên; sắp xếp theo doanh thu giảm dần.

In [5]:
vn_large_orders = (
    df
    .filter(
        (F.col("country") == "VN")
        & (F.col("status") == "Completed")
        & (F.col("net_amount") >= 500)
    )
    .select(
        "order_id",
        "order_date",
        "customer_id",
        "category",
        "quantity",
        "net_amount",
    )
    .orderBy(F.col("net_amount").desc())
)
vn_large_orders.show(20, truncate=False)

+--------+----------+-----------+-----------+--------+----------+
|order_id|order_date|customer_id|category   |quantity|net_amount|
+--------+----------+-----------+-----------+--------+----------+
|25481   |2025-11-19|C009922    |Electronics|5       |7480.65   |
|79525   |2025-02-04|C009124    |Electronics|5       |7476.1    |
|24452   |2025-01-24|C001187    |Electronics|5       |7475.45   |
|77105   |2025-04-02|C003668    |Electronics|5       |7468.7    |
|9217    |2025-02-28|C002105    |Electronics|5       |7467.6    |
|38984   |2025-06-26|C011354    |Electronics|5       |7461.5    |
|43592   |2025-06-27|C012849    |Electronics|5       |7447.05   |
|43228   |2025-04-16|C016992    |Electronics|5       |7443.6    |
|85100   |2025-08-17|C001309    |Electronics|5       |7418.85   |
|26023   |2025-02-03|C006779    |Electronics|5       |7416.85   |
|98198   |2025-02-02|C011410    |Electronics|5       |7396.6    |
|11341   |2025-02-24|C009549    |Electronics|5       |7384.4    |
|73370   |

## 6. Tổng hợp theo category

Tính số đơn hàng hoàn thành, số sản phẩm bán, tổng doanh thu và giá trị đơn hàng trung bình.

In [6]:
revenue_by_category = (
    df
    .filter(F.col("status") == "Completed")
    .groupBy("category")
    .agg(
        F.count("*").alias("completed_orders"),
        F.sum("quantity").alias("items_sold"),
        F.round(F.sum("net_amount"), 2).alias("revenue"),
        F.round(F.avg("net_amount"), 2).alias("avg_order_value"),
    )
    .orderBy(F.col("revenue").desc())
)
revenue_by_category.show(truncate=False)

+-----------+----------------+----------+-------------+---------------+
|category   |completed_orders|items_sold|revenue      |avg_order_value|
+-----------+----------------+----------+-------------+---------------+
|Electronics|16197           |48615     |3.546583942E7|2189.65        |
|Sports     |16487           |49398     |1.407834136E7|853.91         |
|Home       |16453           |49276     |1.192905099E7|725.04         |
|Fashion    |16328           |48673     |6991988.15   |428.22         |
|Books      |16470           |49440     |1936471.88   |117.58         |
+-----------+----------------+----------+-------------+---------------+



## 7. Tổng hợp theo quốc gia và tháng

`countDistinct` đếm khách hàng duy nhất. Các bài tập tự kiểm tra được tính ngay bên dưới; hãy ghi câu trả lời/nhận xét của bạn vào Markdown.

In [7]:
monthly_country = (
    df
    .filter(F.col("status") == "Completed")
    .groupBy("country", "order_month")
    .agg(
        F.countDistinct("customer_id").alias("unique_customers"),
        F.count("*").alias("orders"),
        F.round(F.sum("net_amount"), 2).alias("revenue"),
    )
    .orderBy("country", "order_month")
)
monthly_country.show(100, truncate=False)

+-------+-----------+----------------+------+----------+
|country|order_month|unique_customers|orders|revenue   |
+-------+-----------+----------------+------+----------+
|ID     |1          |1020            |1036  |875986.95 |
|ID     |2          |919             |935   |810099.01 |
|ID     |3          |999             |1028  |864159.45 |
|ID     |4          |980             |1005  |903845.95 |
|ID     |5          |1058            |1078  |965499.17 |
|ID     |6          |892             |916   |854228.69 |
|ID     |7          |997             |1022  |890950.55 |
|ID     |8          |993             |1014  |901161.77 |
|ID     |9          |911             |930   |803276.71 |
|ID     |10         |1016            |1040  |858366.62 |
|ID     |11         |917             |940   |826207.38 |
|ID     |12         |1019            |1045  |899477.78 |
|MY     |1          |1030            |1059  |908432.96 |
|MY     |2          |891             |919   |782953.78 |
|MY     |3          |1018      

In [8]:
# Q1: Quốc gia nào có doanh thu Completed cao nhất?
revenue_by_country = (
    df.filter(F.col("status") == "Completed")
    .groupBy("country")
    .agg(F.round(F.sum("net_amount"), 2).alias("revenue"))
    .orderBy(F.col("revenue").desc())
)
revenue_by_country.show(1, truncate=False)

# Q2: Tháng nào có nhiều customer duy nhất nhất tại VN?
vn_customers_by_month = (
    df.filter((F.col("country") == "VN") & (F.col("status") == "Completed"))
    .groupBy("order_month")
    .agg(F.countDistinct("customer_id").alias("unique_customers"))
    .orderBy(F.col("unique_customers").desc())
)
vn_customers_by_month.show(1, truncate=False)

# Q3: Tỷ lệ đơn Cancelled của từng quốc gia (trên tổng số đơn của quốc gia).
cancelled_rate_by_country = (
    df.groupBy("country")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum(F.when(F.col("status") == "Cancelled", 1).otherwise(0))
        .alias("cancelled_orders"),
    )
    .withColumn(
        "cancelled_rate_pct",
        F.round(F.col("cancelled_orders") / F.col("total_orders") * 100, 2),
    )
    .orderBy("country")
)
cancelled_rate_by_country.show(truncate=False)

+-------+-------------+
|country|revenue      |
+-------+-------------+
|VN     |3.177967531E7|
+-------+-------------+
only showing top 1 row
+-----------+----------------+
|order_month|unique_customers|
+-----------+----------------+
|7          |3000            |
+-----------+----------------+
only showing top 1 row
+-------+------------+----------------+------------------+
|country|total_orders|cancelled_orders|cancelled_rate_pct|
+-------+------------+----------------+------------------+
|ID     |14713       |879             |5.97              |
|MY     |15299       |902             |5.9               |
|SG     |9996        |574             |5.74              |
|TH     |14857       |906             |6.1               |
|VN     |45135       |2747            |6.09              |
+-------+------------+----------------+------------------+



### TODO — Diễn giải kết quả

- Quốc gia có doanh thu `Completed` cao nhất là: **...**
- Tháng có nhiều khách hàng duy nhất nhất tại Việt Nam là: **...**
- Tỷ lệ `Cancelled` thay đổi như thế nào giữa các quốc gia? **...**

Thay dấu `...` bằng kết quả quan sát từ các bảng trên.

## 8. Window Function — top 3 sản phẩm mỗi category

In [9]:
from pyspark.sql.window import Window

product_revenue = (
    df.filter(F.col("status") == "Completed")
    .groupBy("category", "product_id")
    .agg(F.round(F.sum("net_amount"), 2).alias("revenue"))
)

window_spec = Window.partitionBy("category").orderBy(F.col("revenue").desc())
top_products = (
    product_revenue
    .withColumn("rank", F.row_number().over(window_spec))
    .filter(F.col("rank") <= 3)
    .orderBy("category", "rank")
)
top_products.show(100, truncate=False)

+-----------+----------+---------+----+
|category   |product_id|revenue  |rank|
+-----------+----------+---------+----+
|Books      |P0493     |6328.17  |1   |
|Books      |P0015     |6266.99  |2   |
|Books      |P0338     |6046.52  |3   |
|Electronics|P0228     |132717.31|1   |
|Electronics|P0062     |115457.27|2   |
|Electronics|P0011     |114038.19|3   |
|Fashion    |P0110     |24874.9  |1   |
|Fashion    |P0225     |23841.83 |2   |
|Fashion    |P0353     |22630.82 |3   |
|Home       |P0203     |40113.97 |1   |
|Home       |P0268     |39447.93 |2   |
|Home       |P0379     |38050.02 |3   |
|Sports     |P0329     |50435.22 |1   |
|Sports     |P0046     |43120.82 |2   |
|Sports     |P0170     |42866.5  |3   |
+-----------+----------+---------+----+



## 9. Quan sát execution plan

Spark thực thi transformations theo lazy evaluation. `explain("formatted")` cho thấy các operator trong physical plan.

In [10]:
revenue_by_category.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan csv  (1)


(1) Scan csv 
Output [5]: [category#6, quantity#8, unit_price#9, discount_pct#10, status#12]
Batched: false
Location: InMemoryFileIndex [file:/home/jovyan/work/data/sales_100k.csv]
PushedFilters: [IsNotNull(status), EqualTo(status,Completed)]
ReadSchema: struct<category:string,quantity:int,unit_price:double,discount_pct:int,status:string>

(2) Filter
Input [5]: [category#6, quantity#8, unit_price#9, discount_pct#10, status#12]
Condition : (isnotnull(status#12) AND (status#12 = Completed))

(3) Project
Output [3]: [category#6, quantity#8, round((round((cast(quantity#8 as double) * unit_price#9), 2) * (1.0 - (cast(discount_pct#10 as double) / 100.0))), 2) AS net_amount#86]
Input [5]: [category#6, quantity#8, unit_price#9, discount_p

### TODO — Nhận xét execution plan

Dựa trên plan vừa in, ghi lại:

1. Operator nào tạo shuffle (`Exchange`)?
2. `Filter` được thực hiện trước hay sau aggregation?
3. Spark đọc những cột nào và có tối ưu projection/filter ra sao?

**Trả lời:** ...

## 10. Ghi Parquet không partition

Spark ghi một thư mục gồm một hay nhiều file `part-*.parquet`, không phải một file duy nhất.

In [11]:
unpartitioned_path = OUTPUT_DIR / "sales_parquet_unpartitioned"
(
    df.write
    .mode("overwrite")
    .parquet(str(unpartitioned_path))
)
print(f"Wrote unpartitioned Parquet to {unpartitioned_path}")

Wrote unpartitioned Parquet to /home/jovyan/work/output/sales_parquet_unpartitioned


## 11. Ghi Parquet partition theo năm/tháng

Output partitioned được lưu riêng để dễ đọc lại và quan sát partition pruning.

In [12]:
partitioned_path = OUTPUT_DIR / "sales_parquet"
(
    df.write
    .mode("overwrite")
    .partitionBy("order_year", "order_month")
    .parquet(str(partitioned_path))
)
print(f"Wrote partitioned Parquet to {partitioned_path}")

Wrote partitioned Parquet to /home/jovyan/work/output/sales_parquet


## 12. Đọc lại Parquet và kiểm tra số dòng

In [13]:
df_parquet = spark.read.parquet(str(partitioned_path))
df_parquet.printSchema()
parquet_count = df_parquet.count()
print(f"Parquet rows: {parquet_count:,}")
assert parquet_count == row_count

root
 |-- order_id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- status: string (nullable = true)
 |-- gross_amount: double (nullable = true)
 |-- net_amount: double (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)

Parquet rows: 100,000


## 13. Truy vấn partition tháng 8 năm 2025

In [14]:
august_2025 = df_parquet.filter(
    (F.col("order_year") == 2025) & (F.col("order_month") == 8)
)
print(f"August 2025 rows: {august_2025.count():,}")

august_revenue_by_category = (
    august_2025
    .filter(F.col("status") == "Completed")
    .groupBy("category")
    .agg(F.round(F.sum("net_amount"), 2).alias("revenue"))
    .orderBy(F.col("revenue").desc())
)
august_revenue_by_category.show(truncate=False)

August 2025 rows: 8,381
+-----------+----------+
|category   |revenue   |
+-----------+----------+
|Electronics|2798540.55|
|Sports     |1185488.16|
|Home       |1005529.41|
|Fashion    |576232.96 |
|Books      |166348.85 |
+-----------+----------+



## 14. Kiểm tra partition pruning

Tìm `PartitionFilters` trong plan. Nó cho thấy Spark có thể chỉ đọc partition khớp với năm/tháng được lọc.

In [15]:
august_2025.explain("formatted")

== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [17]: [order_id#452L, order_date#453, customer_id#454, customer_segment#455, country#456, city#457, category#458, product_id#459, quantity#460, unit_price#461, discount_pct#462, payment_method#463, status#464, gross_amount#465, net_amount#466, order_year#467, order_month#468]
Batched: true
Location: InMemoryFileIndex [file:/home/jovyan/work/output/sales_parquet]
PartitionFilters: [isnotnull(order_year#467), isnotnull(order_month#468), (order_year#467 = 2025), (order_month#468 = 8)]
ReadSchema: struct<order_id:bigint,order_date:date,customer_id:string,customer_segment:string,country:string,city:string,category:string,product_id:string,quantity:int,unit_price:double,discount_pct:int,payment_method:string,status:string,gross_amount:double,net_amount:double>

(2) ColumnarToRow [codegen id : 1]
Input [17]: [order_id#452L, order_date#453, customer_id#454, customer_segment#455, country#456, city#457, categ

### TODO — Nhận xét partition pruning

Trong plan trên, `PartitionFilters` có những điều kiện nào? Giải thích vì sao truy vấn tháng 8 có thể tránh đọc các partition của những tháng khác.

**Trả lời:** `PartitionFilters` gồm `isnotnull(order_year)`, `isnotnull(order_month)`, `order_year = 2025` và `order_month = 8`. Hai điều kiện `isnotnull` loại các partition không có giá trị năm/tháng; hai điều kiện còn lại giới hạn việc đọc vào partition của tháng 8 năm 2025. Vì dữ liệu được ghi theo thư mục `order_year` và `order_month`, Spark có thể bỏ qua các thư mục thuộc tháng hoặc năm khác — đây là partition pruning. `ReadSchema` không gồm hai cột partition vì Spark lấy giá trị của chúng từ đường dẫn thư mục thay vì đọc từ từng file Parquet.

## Hoàn tất

Sau khi đã lưu notebook và hoàn thành các nhận xét TODO, có thể dừng Spark session:

In [16]:
spark.stop()